In [ ]:
#AKT
import heapq
import random
import time
from math import floor

# ---------------------------------------------------------------
# 1. TRẠNG THÁI PUZZLE
# ---------------------------------------------------------------
class State:
    def __init__(self, board, n, parent=None, move=None, g=0):
        self.board  = board
        self.n      = n
        self.parent = parent
        self.move   = move
        self.g      = g
        self.h      = self._manhattan()
        self.f      = self.g + self.h
    def _manhattan(self):
        dist = 0
        for idx, val in enumerate(self.board):
            if val != 0:
                goal_r = (val - 1) // self.n
                goal_c = (val - 1) %  self.n
                cur_r  = idx // self.n
                cur_c  = idx %  self.n
                dist  += abs(goal_r - cur_r) + abs(goal_c - cur_c)
        return dist
    def is_goal(self):
        return self.board == list(range(1, self.n * self.n)) + [0]
    def get_neighbors(self):
        neighbors = []
        z = self.board.index(0)
        r, c = z // self.n, z % self.n
        directions = {'UP':(-1,0), 'DOWN':(1,0), 'LEFT':(0,-1), 'RIGHT':(0,1)}
        for mv, (dr, dc) in directions.items():
            nr, nc = r + dr, c + dc
            if 0 <= nr < self.n and 0 <= nc < self.n:
                new_board = self.board[:]
                ni = nr * self.n + nc
                new_board[z], new_board[ni] = new_board[ni], new_board[z]
                # cost(S→N) = 1 (mỗi bước di chuyển có chi phí 1)
                neighbors.append(State(new_board, self.n, self, mv, self.g + 1))
        return neighbors

    def __lt__(self, other):
        return self.f < other.f

    def __hash__(self):
        return hash(tuple(self.board))

    def __eq__(self, other):
        return self.board == other.board


# ---------------------------------------------------------------
# 2. THUẬT GIẢI AKT
# ---------------------------------------------------------------

def AKT(start_board, n):
    """
    Giải N-puzzle theo đúng thuật giải AKT trong tài liệu.
    Trả về (danh_sách_bước, số_đỉnh_đã_đóng, thời_gian)
    """
    t0 = time.time()

    # --- Bước 1: Mở đỉnh đầu tiên S ---
    S = State(start_board, n)
    if S.is_goal():
        return [], 0, 0.0

    open_dict  = {tuple(S.board): S}
    closed_set = set()
    nodes_closed = 0

    while True:
        if not open_dict:
            return None, nodes_closed, time.time() - t0  # Thất bại

        min_f = min(s.f for s in open_dict.values())

        candidates = [s for s in open_dict.values() if s.f == min_f]
        goal_candidate = next((s for s in candidates if s.is_goal()), None)
        if goal_candidate:
            path = _trace_path(goal_candidate)
            return path, nodes_closed, time.time() - t0

        N = candidates[0] if len(candidates) == 1 else random.choice(candidates)
        del open_dict[tuple(N.board)]
        closed_set.add(tuple(N.board))
        nodes_closed += 1

        for S_next in N.get_neighbors():
            key = tuple(S_next.board)
            if key in closed_set:
                continue

            if key not in open_dict or S_next.g < open_dict[key].g:
                open_dict[key] = S_next

def _trace_path(goal_state):
    """Truy vết đường đi từ đỉnh đích về đỉnh gốc."""
    path = []
    node = goal_state
    while node.parent is not None:
        path.append(node.move)
        node = node.parent
    path.reverse()
    return path
def is_solvable(board, n):
    flat = [x for x in board if x != 0]
    inv  = sum(1 for i in range(len(flat))
                 for j in range(i+1, len(flat)) if flat[i] > flat[j])
    if n % 2 == 1:
        return inv % 2 == 0
    else:
        blank_row_from_bottom = n - (board.index(0) // n)
        return (inv % 2 == 1) if (blank_row_from_bottom % 2 == 0) else (inv % 2 == 0)

def generate_puzzle(n, shuffles=40):
    """Sinh puzzle bằng cách trộn ngẫu nhiên từ trạng thái đích."""
    board = list(range(1, n*n)) + [0]
    for _ in range(shuffles):
        z = board.index(0)
        r, c = z // n, z % n
        moves = []
        for dr, dc in [(-1,0),(1,0),(0,-1),(0,1)]:
            nr, nc = r+dr, c+dc
            if 0 <= nr < n and 0 <= nc < n:
                moves.append(nr*n + nc)
        ni = random.choice(moves)
        board[z], board[ni] = board[ni], board[z]
    return board


# ---------------------------------------------------------------
# 5. HIỂN THỊ BẢNG
# ---------------------------------------------------------------

def print_board(board, n, title=""):
    if title:
        print(f"\n{title}")
    w   = len(str(n*n))
    sep = "+" + ("-"*(w+2) + "+") * n
    print(sep)
    for row in range(n):
        cells = []
        for col in range(n):
            v = board[row*n + col]
            s = str(v) if v != 0 else " "
            cells.append(s.rjust(w))
        print("| " + " | ".join(cells) + " |")
        print(sep)


def show_solution_steps(start_board, moves, n, max_show=5):
    """Hiển thị tối đa max_show bước đầu và bước cuối."""
    board = start_board[:]
    print_board(board, n, "🔢 Trạng thái ban đầu")
    total = len(moves)
    for i, mv in enumerate(moves):
        z  = board.index(0)
        r, c = z // n, z % n
        dr, dc = {'UP':(-1,0),'DOWN':(1,0),'LEFT':(0,-1),'RIGHT':(0,1)}[mv]
        ni = (r+dr)*n + (c+dc)
        board[z], board[ni] = board[ni], board[z]
        if i < max_show or i == total - 1:
            label = f"Bước {i+1}/{total}: {mv}"
            if i == max_show and total > max_show + 1:
                print(f"\n  ... (bỏ qua {total - max_show - 1} bước giữa) ...")
            print_board(board, n, label)
# ---------------------------------------------------------------
# 6. HÀM CHÍNH
# ---------------------------------------------------------------

def run(n=5, shuffles=15, show_steps=True):
    print(f"\n{'='*52}")
    print(f"  THUẬT GIẢI AKT – {n}×{n} PUZZLE  ({n*n-1}-puzzle)")
    print(f"{'='*52}")

    board = generate_puzzle(n, shuffles)
    assert is_solvable(board, n), "Puzzle không giải được (lỗi sinh puzzle)!"

    print(f"  Kích thước: {n}×{n}   |   Số lần trộn: {shuffles}")
    print_board(board, n, "📌 Trạng thái ban đầu")

    print("\n⏳ Đang chạy thuật giải AKT...")
    moves, closed, elapsed = AKT(board[:], n)

    if moves is None:
        print("❌ Không tìm được lời giải!")
        return

    print(f"\n✅ KẾT QUẢ:")
    print(f"   Số bước giải:        {len(moves)}")
    print(f"   Số đỉnh đã đóng:     {closed:,}")
    print(f"   Thời gian:           {elapsed:.4f} giây")
    print(f"   Trình tự di chuyển:  {' → '.join(moves)}")

    if show_steps:
        show_solution_steps(board, moves, n, max_show=5)

    print(f"\n{'='*52}")
    return moves


# ---------------------------------------------------------------
# 7. CHẠY THỬ NGHIỆM
# ---------------------------------------------------------------

if __name__ == "__main__":

    # --- Ví dụ 1: 5×5 (24-puzzle) – n > 4, trộn ít để giải nhanh ---
    run(n=5, shuffles=12, show_steps=True)

    # --- Ví dụ 2: 6×6 (35-puzzle) – trộn rất ít ---
    run(n=6, shuffles=8, show_steps=False)

    # --- Ví dụ 3: Bảng tùy chỉnh 5×5 ---
    print("\n" + "="*52)
    print("  BẢNG TỰ NHẬP 5×5")
    print("="*52)
    custom = [1,  2,  3,  4,  5,
              6,  7,  8,  9,  10,
              11, 12, 0,  13, 15,
              16, 17, 18, 14, 20,
              21, 22, 23, 19, 24]
    n = 5
    if is_solvable(custom, n):
        print_board(custom, n, "Bảng tùy chỉnh:")
        moves, closed, t = AKT(custom[:], n)
        if moves:
            print(f"✅ Giải được: {len(moves)} bước | {closed} đỉnh đóng | {t:.4f}s")
            print(f"   {' → '.join(moves)}")
    else:
        print("❌ Bảng này không giải được!")



  THUẬT GIẢI AKT – 5×5 PUZZLE  (24-puzzle)
  Kích thước: 5×5   |   Số lần trộn: 12

📌 Trạng thái ban đầu
+----+----+----+----+----+
|  1 |  2 |  3 |  4 |  5 |
+----+----+----+----+----+
|  6 |    |  8 |  9 | 10 |
+----+----+----+----+----+
| 11 |  7 | 13 | 14 | 15 |
+----+----+----+----+----+
| 16 | 12 | 18 | 19 | 20 |
+----+----+----+----+----+
| 21 | 17 | 22 | 23 | 24 |
+----+----+----+----+----+

⏳ Đang chạy thuật giải AKT...

✅ KẾT QUẢ:
   Số bước giải:        6
   Số đỉnh đã đóng:     6
   Thời gian:           0.0005 giây
   Trình tự di chuyển:  DOWN → DOWN → DOWN → RIGHT → RIGHT → RIGHT

🔢 Trạng thái ban đầu
+----+----+----+----+----+
|  1 |  2 |  3 |  4 |  5 |
+----+----+----+----+----+
|  6 |    |  8 |  9 | 10 |
+----+----+----+----+----+
| 11 |  7 | 13 | 14 | 15 |
+----+----+----+----+----+
| 16 | 12 | 18 | 19 | 20 |
+----+----+----+----+----+
| 21 | 17 | 22 | 23 | 24 |
+----+----+----+----+----+

Bước 1/6: DOWN
+----+----+----+----+----+
|  1 |  2 |  3 |  4 |  5 |
+----+----